In [9]:
import pandas as pd

df = pd.read_csv("../data/PhiUSIIL_Phishing_URL_Dataset.csv")
print(df.shape)
print(df.columns.tolist())

(235795, 56)
['FILENAME', 'URL', 'URLLength', 'Domain', 'DomainLength', 'IsDomainIP', 'TLD', 'URLSimilarityIndex', 'CharContinuationRate', 'TLDLegitimateProb', 'URLCharProb', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS', 'LineOfCode', 'LargestLineLength', 'HasTitle', 'Title', 'DomainTitleMatchScore', 'URLTitleMatchScore', 'HasFavicon', 'Robots', 'IsResponsive', 'NoOfURLRedirect', 'NoOfSelfRedirect', 'HasDescription', 'NoOfPopup', 'NoOfiFrame', 'HasExternalFormSubmit', 'HasSocialNet', 'HasSubmitButton', 'HasHiddenFields', 'HasPasswordField', 'Bank', 'Pay', 'Crypto', 'HasCopyrightInfo', 'NoOfImage', 'NoOfCSS', 'NoOfJS', 'NoOfSelfRef', 'NoOfEmptyRef', 'NoOfExternalRef', 'label']


In [10]:
df.iloc[:5, :6]

,FILENAME,URL,URLLength,Domain,DomainLength,IsDomainIP
0,521848.txt,https://www.southbankmosaics.com,31,www.southbankmosaics.com,24,0
1,31372.txt,https://www.uni-mainz.de,23,www.uni-mainz.de,16,0
2,597387.txt,https://www.voicefmradio.co.uk,29,www.voicefmradio.co.uk,22,0
3,554095.txt,https://www.sfnmjournal.com,26,www.sfnmjournal.com,19,0
4,151578.txt,https://www.rewildingargentina.org,33,www.rewildingargentina.org,26,0


In [11]:
print(df["label"].value_counts())
print("Duplicate URLs:", df["URL"].duplicated().sum())
print("Missing values:", df.isnull().sum().sum())

label
1    134850
0    100945
Name: count, dtype: int64
Duplicate URLs: 425
Missing values: 0


In [12]:
print(df[["URL", "label"]].sample(10, random_state=1).to_string())

                                      URL  label
71259                 https://www.dkom.hr      1
39785            https://www.loomlove.com      1
43857   https://www.indianbureaucracy.com      1
194428            http://ethereum-uma.com      0
52413    https://www.alephbookcompany.com      1
32513          https://www.handisport.org      1
188432        https://www.cavernas.org.br      1
115394          https://www.rewview.co.kr      1
8887         https://www.thechronicle.com      1
221925        https://www.blackpointt.com      0


In [13]:
d = df.drop_duplicates(subset="URL").copy()
d["phish"] = (d["label"] == 0).astype(int)

d["https"] = d["URL"].str.startswith("https://")
d["www"] = d["URL"].str.contains(r"^https?://www\.")
d["has_path"] = d["URL"].str.replace(r"^https?://[^/]+", "", regex=True).str.len() > 1

for c in ["https", "www", "has_path"]:
    print(d.groupby(c)["phish"].agg(["mean", "size"]).round(3), "\n")

        mean    size
https               
False  1.000   51635
True   0.266  183735 

        mean    size
www                 
False  1.000   58800
True   0.236  176570 

           mean    size
has_path               
False     0.348  206863
True      1.000   28507 



In [14]:
print(d.groupby("phish")["URL"].apply(lambda s: s.str.len().describe()).round(1))

phish       
0      count    134850.0
       mean         27.2
       std           4.8
       min          16.0
       25%          24.0
       50%          27.0
       75%          30.0
       max          58.0
1      count    100520.0
       mean         46.2
       std          61.2
       min          14.0
       25%          26.0
       50%          34.0
       75%          48.0
       max        6097.0
Name: URL, dtype: float64


In [15]:
d["host"] = (d["URL"].str.replace(r"^https?://", "", regex=True)
                      .str.split("/").str[0]
                      .str.replace(r"^www\.", "", regex=True))
print(d.groupby("phish")["host"].apply(lambda s: s.str.len().describe()).round(1))
print(d[d["phish"] == 1]["host"].sample(15, random_state=2).tolist())
print(d[d["phish"] == 0]["host"].sample(15, random_state=2).tolist())

phish       
0      count    134850.0
       mean         15.2
       std           4.8
       min           4.0
       25%          12.0
       50%          15.0
       75%          18.0
       max          46.0
1      count    100520.0
       mean         22.8
       std          12.8
       min           4.0
       25%          14.0
       50%          20.0
       75%          28.0
       max         110.0
Name: host, dtype: float64
['onedrivemail.ga', 'attloginpage33121.square.site', 'droid.fagdns.com', 'asl.info', 'maildinshaakckjnw426.firebaseapp.com', 'nponbd.com', 'h-11av.web.app', 'att-109869.weeblysite.com', 'mydcbvcaxnhadfas.godaddysites.com', 'juriskiana8.firebaseapp.com', 'myflashnet.id', 'maycanbangionz755.com', 'monirshouvo.github.io', 'jh.nowurl.fun', 'kuerennkaycoato-co-jp.kuercnokayccata.zxvtuo.top']
['illusorya.com', 'rockgympro.com', 'foss.events', 'teachers.ab.ca', 'aksaray.bel.tr', 'cf.edu', 'radiofree.org', 'blueoklahoma.org', 'dundalkeagle.com', 'gameconnect.net

In [16]:
import sys; sys.path.append("..")
from src.features import extract_features

for u in ["https://www.paypal.com/signin?x=1", "http://attloginpage33121.square.site", "http://192.168.1.5/login", "https://www.thechronicle.com"]:
    print(u, extract_features(u))

https://www.paypal.com/signin?x=1 {'host_len': 10, 'n_labels': 2, 'n_dots': 1, 'n_hyphens': 0, 'n_digits': 0, 'digit_ratio': 0.0, 'letter_ratio': 0.9, 'longest_label': 6, 'sld_len': 6, 'sld_entropy': 1.9182958340544896, 'tld_len': 3, 'is_ip': 0, 'suspicious_tld': 0, 'hosting_platform': 0, 'has_punycode': 0}
http://attloginpage33121.square.site {'host_len': 29, 'n_labels': 3, 'n_dots': 2, 'n_hyphens': 0, 'n_digits': 5, 'digit_ratio': 0.1724137931034483, 'letter_ratio': 0.7586206896551724, 'longest_label': 17, 'sld_len': 6, 'sld_entropy': 2.584962500721156, 'tld_len': 4, 'is_ip': 0, 'suspicious_tld': 0, 'hosting_platform': 1, 'has_punycode': 0}
http://192.168.1.5/login {'host_len': 11, 'n_labels': 4, 'n_dots': 3, 'n_hyphens': 0, 'n_digits': 8, 'digit_ratio': 0.7272727272727273, 'letter_ratio': 0.0, 'longest_label': 3, 'sld_len': 1, 'sld_entropy': -0.0, 'tld_len': 1, 'is_ip': 1, 'suspicious_tld': 0, 'hosting_platform': 0, 'has_punycode': 0}
https://www.thechronicle.com {'host_len': 16, 'n